# Ejecutar el pipeline de dengue (de los CSV a las métricas de alerta)

Cada sección llama a un comando de `python -m dengue_pipeline` y muestra lo que produjo. Ejecutar **de arriba hacia abajo**.
Estado: todo el bloque `backtest:` de `config/config.yaml` es **PROPUESTA**; el retraso de notificación está aprobado desde el 23-nov-2026 (patrón de Piura simulado en todas las unidades).
**Los resultados de esta corrida no se citan como resultado del artículo.**

Antes de ejecutar, revisar la celda de parámetros (rutas, fechas de descarga y los interruptores).

In [ ]:
import os, subprocess, sys
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# --- Archivos de entrada (descargados a mano; no se versionan) ---
NACIONAL = REPO / "data/raw/nacional.csv"          # CSV nacional de vigilancia de dengue (CDC)
REGIONAL = REPO / "data/raw/piura.csv"             # corte regional de Piura
POBLACION_DISTRITOS_XLSX = REPO / "data/raw/poblacion_distrital_2018_2026.xlsx"  # INEI (gob.pe): Población total proyectada al 30 de junio, por distrito, 2018-2026; para Lima Norte
POBLACION_XLSX = REPO / "data/raw/proy_03_4.xlsx"  # INEI: Población estimada al 30 de junio, por años calendario y sexo, según departamento, 2000-2026

# --- Para el manifiesto (lo declara quien descarga). Mientras diga [FALTA], la celda de registro se omite ---
URL_NACIONAL, URL_REGIONAL, FECHA_DESCARGA = "[FALTA]", "[FALTA]", "[FALTA]"

# --- Interruptores ---
RUN_BACKTEST = os.environ.get("RUN_BACKTEST", "1") == "1"   # ~7 min
MODELOS = os.environ.get("MODELOS") or None                 # None = los cuatro + persistencia; ej. "gb,negbin"
RUN_CLIMA_REAL = False     # requiere red y config/coordenadas.csv completo [FALTA]
RUN_ICEN = False           # requiere config/icen.csv (month,value,published_on,kind) [FALTA]
USAR_EJEMPLO = False       # True = datos SINTÉTICOS de clima/ICEN solo para probar la interfaz (salidas EJEMPLO_NO_REAL)
COORDENADAS = REPO / "config/coordenadas.csv"
ICEN_CSV = REPO / "config/icen.csv"
CLIMA_INICIO, CLIMA_FIN = "20091101", "20241231"   # ventana del clima: estudio 2010-2024 más rezagos (descarga del 23-nov-2026)
FECHA_DESCARGA_CLIMA = "[FALTA]"                   # AAAA-MM-DD, lo declara quien descarga

def run(*args, check=True):
    """Ejecuta `python -m dengue_pipeline ...` desde la raíz del repositorio y muestra la salida."""
    env = {**os.environ, "PYTHONPATH": str(REPO / "src"), "MPLBACKEND": "Agg"}
    r = subprocess.run([sys.executable, "-m", "dengue_pipeline", *map(str, args)], cwd=REPO, env=env, capture_output=True, text=True)
    print(r.stdout[-6000:], r.stderr[-3000:] if r.returncode else "")
    if check and r.returncode:
        raise RuntimeError(f"falló: {' '.join(map(str, args))} (código {r.returncode})")
    return r

for p in (NACIONAL, REGIONAL, POBLACION_XLSX, POBLACION_DISTRITOS_XLSX):
    print(("OK      " if p.exists() else "FALTA   ") + str(p))

## 0. Pruebas
Deben pasar antes de tocar datos reales (los casos con archivos reales se saltan si no se definen las variables `DENGUE_*`).

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-800:]); assert r.returncode == 0

## 1. Registrar los CSV crudos (etapa 1)
Guarda hash SHA-256, filas y fecha de descarga en `data/manifest.json`. La URL y la fecha las declara quien descarga.

In [ ]:
if "[FALTA]" in (URL_NACIONAL, URL_REGIONAL, FECHA_DESCARGA):
    print("Se omite el registro: completar URL_NACIONAL, URL_REGIONAL y FECHA_DESCARGA en la celda de parámetros.")
else:
    run("register", NACIONAL, "--kind", "national", "--source-url", URL_NACIONAL, "--downloaded-on", FECHA_DESCARGA)
    run("register", REGIONAL, "--kind", "regional", "--source-url", URL_REGIONAL, "--downloaded-on", FECHA_DESCARGA)

## 2. Validar (etapa 2)
Termina con error si alguna regla da FALLA; las ADVERTENCIAS no bloquean. Informe en `reports/validation_report.md`.

In [ ]:
run("validate", "--national", NACIONAL, "--regional", REGIONAL)

## 3. Serie final, retraso y datos disponibles (etapas 3 y 4)
`delay` y `simulate` usan el retraso de Piura, aprobado el 23-nov-2026, para todas las unidades.

In [ ]:
run("integrate", "--national", NACIONAL)
run("delay", "--regional", REGIONAL)
run("simulate", "--national", NACIONAL, "--regional", REGIONAL)

## 4. Población, incidencia de 4 semanas y niveles de alerta
Niveles: verde < 50, amarillo 50 a < 150, rojo >= 150 por 100 000 (sensibilidad 100/300). **Lima Norte no tiene población [FALTA]: queda sin tasa.**

In [ ]:
run("population", POBLACION_XLSX, "--districts", POBLACION_DISTRITOS_XLSX)
run("incidence")
inc = pd.read_csv(REPO / "data/processed/weekly_incidence.csv")
display(inc.groupby("unit")["level"].value_counts().unstack(fill_value=0).rename(columns={-1: "sin tasa", 0: "verde", 1: "amarillo", 2: "rojo"}))

## 5. Exploración de datos (EDA)
Figuras y tablas en `reports/`; lectura en `reports/eda_resumen.md`.

In [ ]:
run("eda")
for f in sorted((REPO / "reports").glob("eda_0*.png")):
    display(Image(filename=str(f), width=900))

## 6. Backtesting (etapas 5 y 6), condición `historial`
Dos condiciones de datos (serie final y datos disponibles), horizontes 2 y 4. Tarda unos 7 minutos. Si `RUN_BACKTEST = False` se usan las predicciones que ya existan en `data/processed/backtest/`.

In [ ]:
if RUN_BACKTEST:
    args = ["backtest"] + (["--models", MODELOS] if MODELOS else [])
    run(*args)
else:
    print("RUN_BACKTEST = False: se usan las predicciones existentes.")
print((REPO / "data/processed/backtest/predictions.csv.gz").exists())

## 7. Métricas de alerta sobre las predicciones
Sensibilidad, tasa de falsa alarma, falsas alarmas sobre alertas y anticipación por episodio; IC por bootstrap de bloques de semanas; por año y por unidad; diferencia pareada entre condiciones.
**No se citan ni se publican cifras de desempeño hasta tener la conformidad del profesor.**

In [ ]:
run("alerts")
m = pd.read_csv(REPO / "data/processed/backtest/alert_metrics.csv")
sel = m[(m.agrupacion == "todo") & (m.cortes == "base") & (m.umbral == "amarillo_o_mas")]
display(sel[["condition", "model", "horizon", "tp", "fn", "fp", "tn", "sensibilidad", "sensibilidad_lo", "sensibilidad_hi", "tasa_falsa_alarma"]].round(3))
display(pd.read_csv(REPO / "data/processed/backtest/alert_episode_summary.csv").head(12))

## 8. Clima (NASA POWER) e ICEN: ablación de tres condiciones

Condiciones: `historial`, `clima` (historial + clima local) y `clima_icen` (+ ICEN). Los hiperparámetros son los congelados de `historial`.

**Qué falta traer (todo [FALTA]):**
1. `config/coordenadas.csv` con columnas `unit,lat,lon,source` (partir de `config/coordenadas_plantilla.csv`; la fuente de cada coordenada debe quedar escrita).
2. Red hacia `power.larc.nasa.gov` y la fecha de descarga. La descarga está escrita pero **no se ha ejecutado contra la red [NO VERIFICADO]**.
3. `config/icen.csv` con `month,value,published_on,kind` (`kind` = `definitivo` o `temporal`): una fila por versión publicada de cada mes, con la fecha en que se publicó. Sin ella no existe la condición `clima_icen`.

Con `USAR_EJEMPLO = True` se prueban las tres condiciones con datos **sintéticos**; las salidas llevan el prefijo `EJEMPLO_NO_REAL` y no son resultados.

In [ ]:
if RUN_CLIMA_REAL:
    run("climate", "--coordinates", COORDENADAS, "--start", CLIMA_INICIO, "--end", CLIMA_FIN, "--downloaded-on", FECHA_DESCARGA_CLIMA)
else:
    print("RUN_CLIMA_REAL = False: no se descarga clima.")
if RUN_ICEN:
    run("icen", ICEN_CSV)
else:
    print("RUN_ICEN = False.")

In [ ]:
condiciones = ["clima", "clima_icen"]
for c in condiciones:
    if c == "clima_icen" and not (USAR_EJEMPLO or RUN_ICEN):
        print("Se omite clima_icen: falta la serie del ICEN [FALTA]."); continue
    if c == "clima" and not (USAR_EJEMPLO or RUN_CLIMA_REAL):
        print("Se omite clima: falta el clima real [FALTA]."); continue
    args = ["backtest", "--features", c] + (["--example-exog"] if USAR_EJEMPLO else []) + (["--models", MODELOS] if MODELOS else [])
    run(*args)
print(sorted(p.name for p in (REPO / "data/processed/backtest").glob("*")))

## 9. Fin
Guardar `data/manifest.json`, `config/config.yaml` y la versión del código junto con los resultados. Los valores de `config/config.yaml` son los de la corrida de referencia (ver también `reports/revision_protocolo.md`).